In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
df.shape

(2000, 8)

In [4]:
df.head(3)

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


In [5]:
df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [6]:
df=df.drop('id',axis=1)

In [7]:
answer=df['answer']
df=df.drop('answer',axis=1)

In [8]:
df.shape

(2000, 6)

In [9]:
import re

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^a-zA-Z0-9\s]',' ',text)
    text=re.sub(r'\s+',' ',text).strip()
    return text

df['prompt']=df['prompt'].apply(clean_text)
df['A']=df['A'].apply(clean_text)
df['B']=df['B'].apply(clean_text)
df['C']=df['C'].apply(clean_text)
df['D']=df['D'].apply(clean_text)
df['E']=df['E'].apply(clean_text)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf=TfidfVectorizer(sublinear_tf=True)
tfidf_train=tfidf.fit_transform(df['prompt']+" "+df['A']+" "+df['B']+" "+df['C']+" "+df['D']+" "+df['E'])

In [11]:
tfidf_train.shape

(2000, 2939)

In [12]:
tfidf_scores = []
for col in ['A', 'B', 'C', 'D', 'E']:
    tfidf_options=tfidf.transform(df[col])
    print(tfidf_options.shape)
    scores = np.diag(cosine_similarity(tfidf_train, tfidf_options))
    tfidf_scores.append(scores)

(2000, 2939)
(2000, 2939)
(2000, 2939)
(2000, 2939)
(2000, 2939)


In [13]:
tfidf_scores=np.array(tfidf_scores).T

In [14]:
tfidf.get_feature_names_out()

array(['000', '013', '013343', ..., 'yukawa', 'zero', 'zweig'],
      shape=(2939,), dtype=object)

In [15]:
from gensim.models import Word2Vec
np.random.seed(1)
from numpy.random import seed
seed(1)

all_text=(df['prompt'].tolist()+ df['A'].tolist()+df['B'].tolist()+df['C'].tolist()+df['D'].tolist()+df['E'].tolist())
sentences=[text.split() for text in all_text]
word2vec=Word2Vec(sentences,vector_size=100,window=3,min_count=1,seed=1,workers=1,epochs=10)

In [16]:
def sentence_embedding(text):
    words=text.split()
    embedding=[word2vec.wv[word] for word in words if word in word2vec.wv]
    if len(embedding)==0:
        return np.zeros(word2vec.vector_size)
    return np.mean(embedding,axis=0)

In [17]:
prompts=df['prompt'].apply(sentence_embedding)
prompts=np.vstack(prompts)

option_cols=['A','B','C','D','E']
all_scores=[]

for col in option_cols:
    option_matrix=np.vstack(df[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(prompts,option_matrix))
    all_scores.append(scores)
all_scores=np.array(all_scores).T
print(len(all_scores[0]))

5


In [18]:
all_scores

array([[ 0.68010163,  0.6835704 ,  0.79020584,  0.74668837,  0.63893163],
       [ 0.64669263,  0.6912105 ,  0.70214164,  0.6228468 ,  0.6186186 ],
       [-0.0430439 ,  0.13547257,  0.13956621, -0.00985483, -0.10341667],
       ...,
       [ 0.15629514,  0.170781  ,  0.20528962,  0.17931819,  0.1696489 ],
       [ 0.6563369 ,  0.5649115 ,  0.6594888 ,  0.55413723,  0.50710464],
       [ 0.65726537,  0.65807384,  0.6580738 ,  0.65807384,  0.65726537]],
      shape=(2000, 5), dtype=float32)

In [19]:
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()
final1=ss.fit_transform(all_scores)
final2=ss.fit_transform(tfidf_scores)
final=(final1+final2)/2

In [20]:
labels=[]
for score in final:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65+i) for i in scored]
    labels.append(top_3)

In [21]:
def map_at_3(predictions, ground_truth):
    ap_scores=[1.0 /(pred.index(truth)+1) if truth in pred else 0.0 for pred,truth in zip(predictions, ground_truth)]
    return sum(ap_scores)/len(ap_scores)
print(map_at_3(labels,answer))

0.4795


In [22]:
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

test['prompt']=test['prompt'].apply(clean_text)
test['A']=test['A'].apply(clean_text)
test['B']=test['B'].apply(clean_text)
test['C']=test['C'].apply(clean_text)
test['D']=test['D'].apply(clean_text)
test['E']=test['E'].apply(clean_text)

test_prompts=test['prompt'].apply(sentence_embedding)
test_prompts=np.vstack(test_prompts)

test_all_scores=[]
for col in ['A','B','C','D','E']:
    option_matrix=np.vstack(test[col].apply(sentence_embedding))
    scores=np.diag(cosine_similarity(test_prompts,option_matrix))
    test_all_scores.append(scores)

test_all_scores=np.array(test_all_scores).T

In [23]:
test_labels=[]
for score in test_all_scores:
    scored=np.argsort(score)[::-1][:3]
    top_3=[chr(65 + i) for i in scored]
    test_labels.append(' '.join(top_3))

submission=pd.DataFrame({'id':test['id'],'Prediction':test_labels})
submission.to_csv('submission_model1_w2v.csv',index=False)
print("Model 1 submission saved!")
print(submission.head())

Model 1 submission saved!
   id Prediction
0   1      C A D
1   2      B D E
2   3      D A C
3   4      A C E
4   5      B D C
